# FYP Steganography — Resume-Safe Training Notebook
Run this notebook top to bottom any time: on first run it starts training from scratch;
if a checkpoint already exists (e.g. after a disconnect), it automatically resumes from it.

**First: Runtime -> Change runtime type -> T4 GPU -> Save**

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU - enable the GPU runtime!')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone the repo (fresh each session — local disk is wiped on disconnect)

In [ ]:
import os
REPO_URL = 'https://github.com/syakir12343425/FYP-Project.git'
REPO_DIR = '/content/FYP-Project'

if os.path.exists(REPO_DIR):
    import shutil
    shutil.rmtree(REPO_DIR)

!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pwd
!ls

In [ ]:
!pip install -q pytorch-msssim scikit-image opencv-python-headless pyyaml tqdm pandas

## Patch train.py (AMP/BCE autocast fix) and dataset.py (skip corrupted files)
These fixes aren't in the GitHub repo yet — remove this cell once you've pushed them there.

In [ ]:
# --- patch train.py: compute loss in full precision, outside autocast ---
with open('train.py') as f:
    content = f.read()

old = '''                recovered = decoder(downscaled)
                loss, parts = total_loss(
                    cover, stego, message, recovered,
                    lambda_perceptual=cfg["loss"]["lambda_perceptual"],
                    lambda_mse=cfg["loss"]["lambda_mse"],
                    bce_weight=cfg["loss"]["bce_weight"],
                )'''
new = '''                recovered = decoder(downscaled)

            # loss computed in full precision — BCE after a sigmoid is not
            # autocast-safe, PyTorch errors under mixed precision otherwise
            loss, parts = total_loss(
                cover.float(), stego.float(), message.float(), recovered.float(),
                lambda_perceptual=cfg["loss"]["lambda_perceptual"],
                lambda_mse=cfg["loss"]["lambda_mse"],
                bce_weight=cfg["loss"]["bce_weight"],
            )'''

if old in content:
    content = content.replace(old, new)
    with open('train.py', 'w') as f:
        f.write(content)
    print('train.py patched')
else:
    print('train.py already patched or pattern not found — check manually if training errors')

In [ ]:
# --- patch dataset.py: skip corrupted/missing images instead of crashing ---
with open('dataset.py') as f:
    content = f.read()

old = '''    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")'''
new = '''    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert("RGB")
        except (OSError, Exception):
            return self.__getitem__((idx + 1) % len(self.paths))'''

if old in content:
    content = content.replace(old, new)
    with open('dataset.py', 'w') as f:
        f.write(content)
    print('dataset.py patched')
else:
    print('dataset.py already patched or pattern not found — check manually if training errors')

## Copy DIV2K from Drive to local disk (fast, avoids Drive read lag during training)
If this errors because a split is missing from Drive, go back to the main setup notebook
and re-download that split first.

In [ ]:
import shutil

DATA_DIR = '/content/drive/MyDrive/fyp_data'
LOCAL_DATA = '/content/data'
os.makedirs(LOCAL_DATA, exist_ok=True)

for split, expected in [('DIV2K_train_HR', 800), ('DIV2K_valid_HR', 100)]:
    src = f'{DATA_DIR}/{split}'
    dst = f'{LOCAL_DATA}/{split}'
    assert os.path.exists(src), f"{src} not found on Drive — download it first"
    if not os.path.exists(dst):
        print(f'Copying {split}...')
        shutil.copytree(src, dst)
    n = len(os.listdir(dst))
    print(split, '->', n, 'files', '(OK)' if n == expected else f'(expected {expected}!)')

## Point config at local data + Drive checkpoints (checkpoints MUST stay on Drive to survive disconnects)

In [ ]:
import yaml

CHECKPOINT_DIR = '/content/drive/MyDrive/fyp_checkpoints'

with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['train_dir'] = f'{LOCAL_DATA}/DIV2K_train_HR'
cfg['data']['val_dir'] = f'{LOCAL_DATA}/DIV2K_valid_HR'
cfg['train']['checkpoint_dir'] = CHECKPOINT_DIR

with open('configs/config.yaml', 'w') as f:
    yaml.dump(cfg, f)

print('train_dir:', cfg['data']['train_dir'])
print('val_dir:', cfg['data']['val_dir'])
print('checkpoint_dir:', cfg['train']['checkpoint_dir'])

## Train — auto-resumes if a checkpoint already exists on Drive
This is the cell that makes the notebook "resume-safe": it checks for
`last.pt` on Drive and only adds `--resume` if one is actually there.

In [ ]:
last_ckpt = f'{CHECKPOINT_DIR}/last.pt'

if os.path.exists(last_ckpt):
    print(f'Found existing checkpoint at {last_ckpt} — resuming training.')
    !python train.py --config configs/config.yaml --resume {last_ckpt}
else:
    print('No existing checkpoint found — starting training from scratch.')
    !python train.py --config configs/config.yaml

## If this notebook disconnects mid-training
Just re-run every cell from the top (Runtime -> Run all, or run cells one by one again).
The last cell will detect `last.pt` on Drive and resume automatically — no manual edits needed.

## Full evaluation (run once training is complete / early-stopped)

In [ ]:
RESULTS_DIR = '/content/drive/MyDrive/fyp_results'
!python eval.py --config configs/config.yaml --checkpoint {CHECKPOINT_DIR}/best.pt --out-dir {RESULTS_DIR}

In [ ]:
import pandas as pd
summary = pd.read_csv(f'{RESULTS_DIR}/eval_summary.csv')
summary